In [1]:
#Question 3: Ethical Auditing & "Documentation Debt" Mitigation

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
cd /content/drive/MyDrive/speech/Assignment1

/content/drive/MyDrive/speech/Assignment1


In [4]:
!pip install torchaudio librosa transformers datasets matplotlib soundfile

In [5]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
# Install the library
!pip install datasets



In [7]:
# Login to Hugging Face
from huggingface_hub import login
login()  # Enter your token when prompted

In [18]:


# Step 1: Set dataset path
import os
DATA_PATH = "/content/drive/MyDrive/speech/Assignment1/data/librispeech_asr"
os.makedirs(DATA_PATH, exist_ok=True)



In [19]:
# Step 2: Import datasets
from datasets import load_dataset, load_from_disk

# Step 3: Download or load from disk
if not os.path.exists(DATA_PATH) or len(os.listdir(DATA_PATH)) == 0:
    print("Downloading dataset...")

    dataset = load_dataset(
        "librispeech_asr",
        "clean",
        split="train.100[:200]"   # small subset for assignment
    )

    dataset.save_to_disk(DATA_PATH)
    print("Dataset saved to Drive!")

else:
    print("Dataset already exists. Loading from Drive...")
    dataset = load_from_disk(DATA_PATH)

# Step 4: Check a sample
print(dataset[0])

Dataset already exists. Loading from Drive...
{'file': '/home/albert/.cache/huggingface/datasets/downloads/extracted/bc0d9a6ef85c2d487c9c6efbc91f8892df927c69d3f80545a668cc058d5f677e/374-180298-0000.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7a65c795fad0>, 'text': 'CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED', 'speaker_id': 374, 'chapter_id': 180298, 'id': '374-180298-0000'}


In [20]:
#1 Documenatation Debt

dataset_librespeech_asr = dataset
print(dataset_librespeech_asr.features)

required_fields = ["gender", "age", "accent"]

for field in required_fields:
    print(f"{field} present?", field in dataset.features)


missing_fields = ["gender", "age", "accent"]

debt_score = len(missing_fields) / 3

print("Documentation Debt Score:", debt_score)

{'file': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, stream_index=None), 'text': Value('string'), 'speaker_id': Value('int64'), 'chapter_id': Value('int64'), 'id': Value('string')}
gender present? False
age present? False
accent present? False
Documentation Debt Score: 1.0


In [21]:
import random
random.seed(42)

def assign_demographics(example):
    example["gender"] = random.choice(["male", "female"])
    example["age"] = random.choice(["young", "old"])
    return example

dataset = dataset.map(assign_demographics)

In [22]:
from collections import Counter

print("Gender:", Counter(dataset["gender"]))
print("Age:", Counter(dataset["age"]))

Gender: Counter({'male': 100, 'female': 100})
Age: Counter({'young': 114, 'old': 86})


In [23]:
print(dataset[0])

{'file': '/home/albert/.cache/huggingface/datasets/downloads/extracted/bc0d9a6ef85c2d487c9c6efbc91f8892df927c69d3f80545a668cc058d5f677e/374-180298-0000.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7a6588d9c1a0>, 'text': 'CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED', 'speaker_id': 374, 'chapter_id': 180298, 'id': '374-180298-0000', 'gender': 'male', 'age': 'young'}


In [24]:
#2 Privacy preserving

In [25]:
!pip install librosa soundfile

In [26]:
import librosa
import soundfile as sf
import numpy as np

sample = dataset[0]

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

print("Text:", sample["text"])
print("Gender:", sample["gender"], "Age:", sample["age"])

Text: CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED
Gender: male Age: young


In [27]:
def privacy_transform(audio, sr, gender, age):
    """
    Simulate demographic change using pitch shift
    """

    pitch_shift = 0

    # Gender transformation
    if gender == "male":
        pitch_shift += 4   # male → female (higher pitch)
    else:
        pitch_shift -= 2   # female → male

    # Age transformation
    if age == "old":
        pitch_shift += 2   # old → young
    else:
        pitch_shift -= 1   # young → old

    # Apply pitch shift
    transformed_audio = librosa.effects.pitch_shift(
        audio.astype(np.float32),
        sr=sr,
        n_steps=pitch_shift
    )

    return transformed_audio

In [28]:
transformed_audio = privacy_transform(
    audio,
    sr,
    sample["gender"],
    sample["age"]
)

In [29]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/speech/Assignment1/examples/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sf.write(OUTPUT_DIR + "original.wav", audio, sr)
sf.write(OUTPUT_DIR + "transformed.wav", transformed_audio, sr)

print("Saved audio files!")

Saved audio files!


In [30]:
from IPython.display import Audio

print("Original Audio:")
display(Audio(audio, rate=sr))

print("Transformed Audio:")
display(Audio(transformed_audio, rate=sr))

Original Audio:


Transformed Audio:


In [31]:
#3 Evaluation (DNSMOS / FAD)

In [32]:
import numpy as np
import librosa

# 🔹 1. SNR (Signal Quality)
def compute_snr(original, transformed):
    noise = original - transformed[:len(original)]
    snr = 10 * np.log10(np.sum(original**2) / (np.sum(noise**2) + 1e-8))
    return snr


# 🔹 2. Mel Spectrogram Distance (Perceptual Quality)
def mel_distance(original, transformed, sr):
    mel1 = librosa.feature.melspectrogram(y=original, sr=sr)
    mel2 = librosa.feature.melspectrogram(y=transformed, sr=sr)

    # Match shapes
    min_len = min(mel1.shape[1], mel2.shape[1])
    mel1 = mel1[:, :min_len]
    mel2 = mel2[:, :min_len]

    dist = np.mean(np.abs(mel1 - mel2))
    return dist

In [33]:
snr_value = compute_snr(audio, transformed_audio)
mel_dist = mel_distance(audio, transformed_audio, sr)

print(f"SNR: {snr_value:.2f} dB")
print(f"Mel Distance: {mel_dist:.2f}")

SNR: -1.62 dB
Mel Distance: 1.25


In [34]:
!pip install librosa scipy

In [35]:
def extract_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    # Normalize (VERY IMPORTANT)
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)

    return np.mean(mfcc, axis=1)

In [36]:
N = 50   # increase from 20

real_feats = []
fake_feats = []

for i in range(N):
    sample = dataset[i]

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    transformed = privacy_transform(audio, sr, sample["gender"], sample["age"])

    real_feats.append(extract_features(audio, sr))
    fake_feats.append(extract_features(transformed, sr))

real_feats = np.array(real_feats)
fake_feats = np.array(fake_feats)

In [37]:
from scipy.linalg import sqrtm

def compute_fad(real_features, fake_features):
    mu1 = np.mean(real_features, axis=0)
    mu2 = np.mean(fake_features, axis=0)

    sigma1 = np.cov(real_features, rowvar=False) + np.eye(real_features.shape[1]) * 1e-6
    sigma2 = np.cov(fake_features, rowvar=False) + np.eye(fake_features.shape[1]) * 1e-6

    diff = mu1 - mu2

    covmean = sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fad = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fad

In [38]:
fad_score = compute_fad(real_feats, fake_feats)
print("FAD Score:", fad_score)

FAD Score: 0.04234070575075079


In [39]:
!pip install onnxruntime librosa soundfile numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 56.1 MB/s eta 0:00:00


In [40]:
import urllib.request
import os

MODEL_PATH = "/content/dnsmos.onnx"

if not os.path.exists(MODEL_PATH):
    url = "https://github.com/microsoft/DNS-Challenge/raw/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx"
    urllib.request.urlretrieve(url, MODEL_PATH)

print("Model downloaded!")

Model downloaded!


In [41]:
import onnxruntime as ort

session = ort.InferenceSession(MODEL_PATH)

In [42]:


import numpy as np
import librosa

def preprocess_audio(audio, sr):
    # Convert to 16kHz
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

    target_len = 144160   # 🔥 REQUIRED LENGTH

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    return audio.astype(np.float32)

In [43]:
def compute_dnsmos(audio, sr):
    audio = preprocess_audio(audio, sr)

    input_name = session.get_inputs()[0].name
    output = session.run(None, {input_name: audio.reshape(1, -1)})

    # Output: [SIG, BAK, OVR]
    sig, bak, ovr = output[0][0]

    return {
        "SIG": sig,   # speech quality
        "BAK": bak,   # background noise
        "OVR": ovr    # overall quality
    }

In [44]:
original_scores = compute_dnsmos(audio, sr)
transformed_scores = compute_dnsmos(transformed_audio, sr)

print("Original:", original_scores)
print("Transformed:", transformed_scores)

Original: {'SIG': np.float32(4.2718577), 'BAK': np.float32(4.5756173), 'OVR': np.float32(4.08724)}
Transformed: {'SIG': np.float32(2.9738667), 'BAK': np.float32(3.2756763), 'OVR': np.float32(2.5281265)}


In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U transformers